# Prueba de Concepto (PoC) V4.0 - TFM Fases 1 y 2

Este notebook implementa las dos primeras fases de la arquitectura híbrida GNN-HVA para la caracterización de fases topológicas, utilizando el Modelo de Ising en Campo Transversal (TFIM) 1D como sistema de prueba.

**Correcciones respecto a V1:**
- Función de costo basada en energía (StatevectorEstimator), no fidelidad global
- Factor `2*theta` en puertas RZZ/RX para correspondencia física correcta
- Diagonalización exacta densa (`np.linalg.eigh`) en lugar de `eigsh`
- Observables promediados sobre todos los sitios
- Almacenamiento de energías, gaps y persistencia de datos

## Celda 1: Importaciones y Configuración Inicial

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from qiskit.circuit import ParameterVector, QuantumCircuit
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp, Statevector, state_fidelity
from scipy.optimize import minimize

# Reproducibilidad
np.random.seed(42)

# Configuración del modelo
N = 6
J = 1.0
# Non-uniform grid: denser sampling near the finite-size critical point (~1.1)
# where θ_opt landscape changes most abruptly (Miao et al., NN-VQE, 2024).
h_coarse = np.arange(0.0, 0.8, 0.1)          # ferromagnetic: Δh=0.1
h_dense  = np.arange(0.8, 1.45, 0.05)        # critical region: Δh=0.05
h_coarse2 = np.arange(1.5, 2.05, 0.1)        # paramagnetic: Δh=0.1
h_values = np.unique(np.concatenate([h_coarse, h_dense, h_coarse2]))
p_layers = 2

print(f"TFIM 1D: N={N}, J={J}, p={p_layers}, h ∈ [{h_values[0]}, {h_values[-1]}]")

## Celda 2: Construcción del Hamiltoniano y Observables

In [ ]:
def build_tfim_hamiltonian(N, J, h):
    """H = -J Σ ZᵢZᵢ₊₁ - h Σ Xᵢ"""
    terms = []
    for i in range(N - 1):
        terms.append(("ZZ", [i, i + 1], -J))
    for i in range(N):
        terms.append(("X", [i], -h))
    return SparsePauliOp.from_sparse_list(terms, num_qubits=N)


# Observables locales por sitio
ops_X = [SparsePauliOp.from_sparse_list([("X", [i], 1.0)], num_qubits=N) for i in range(N)]
ops_ZZ = [SparsePauliOp.from_sparse_list([("ZZ", [i, i+1], 1.0)], num_qubits=N) for i in range(N - 1)]

print(f"Observables: {N} sitios ⟨Xᵢ⟩, {N-1} enlaces ⟨ZᵢZᵢ₊₁⟩")

## Celda 3: FASE 1 — Diagonalización Exacta y Ground Truth

In [ ]:
exact_data = []

print("Fase 1: Diagonalización exacta...")
for h in h_values:
    H = build_tfim_hamiltonian(N, J, h)
    evals, evecs = np.linalg.eigh(H.to_matrix())
    psi_gs = np.ascontiguousarray(evecs[:, 0])
    sv = Statevector(psi_gs)

    mag_x = np.mean([sv.expectation_value(op).real for op in ops_X])
    corr_zz = np.mean([sv.expectation_value(op).real for op in ops_ZZ])

    exact_data.append({
        "h": h, "ground_energy": evals[0], "gap": evals[1] - evals[0],
        "ground_state": psi_gs, "mag_x": mag_x, "corr_zz": corr_zz,
    })

# Visualización
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(h_values, [d["corr_zz"] for d in exact_data], 'o-', label=r'$\langle Z_i Z_{i+1} \rangle$ (avg)')
ax1.plot(h_values, [d["mag_x"] for d in exact_data], 's-', label=r'$\langle X_i \rangle$ (avg)')
ax1.axvline(x=1.0, color='red', linestyle='--', label='Punto Crítico')
ax1.set_xlabel('h/J'); ax1.set_ylabel('Valor Esperado')
ax1.set_title('Parámetros de Orden'); ax1.legend(); ax1.grid(True)

ax2.plot(h_values, [d["gap"] for d in exact_data], 'D-', color='purple')
ax2.axvline(x=1.0, color='red', linestyle='--', label='Punto Crítico')
ax2.set_xlabel('h/J'); ax2.set_ylabel('Δ = E₁ - E₀')
ax2.set_title('Gap Espectral'); ax2.legend(); ax2.grid(True)

plt.tight_layout(); plt.show()
print(f"Fase 1 completada. Gap mínimo: {min(d['gap'] for d in exact_data):.6f}")

## Celda 4: FASE 2 (Parte A) — Ansatz HVA con factor 2θ correcto

In [ ]:
def create_hva_circuit(N, p):
    """HVA: e^{-iθ_x H_X} · e^{-iθ_zz H_ZZ} por capa sobre |+⟩^N. Factor 2θ para RZZ/RX."""
    qc = QuantumCircuit(N)
    # Estado inicial |+⟩^N: ground state paramagnético (h → ∞)
    qc.h(range(N))
    theta = ParameterVector('θ', 2 * p)

    for layer in range(p):
        for i in range(N - 1):
            qc.rzz(2 * theta[layer * 2], i, i + 1)
        for i in range(N):
            qc.rx(2 * theta[layer * 2 + 1], i)

    return qc, theta


hva_qc, theta_params = create_hva_circuit(N, p_layers)
print(f"HVA: {hva_qc.num_parameters} parámetros, profundidad p={p_layers}")
try:
    display(hva_qc.draw('mpl'))
except Exception:
    print(hva_qc.draw('text'))

## Celda 5: FASE 2 (Parte B) — VQE con Warm Start y costo energético

In [ ]:
estimator = StatevectorEstimator()
vqe_results_dict = {}


def cost_fn(params, hamiltonian):
    """Energía VQE: ⟨ψ(θ)|H|ψ(θ)⟩"""
    bound = hva_qc.assign_parameters(params)
    return float(estimator.run([(bound, hamiltonian)]).result()[0].data.evs)


def multi_start_vqe(hamiltonian, initial_guess, n_restarts=3):
    """Multi-start L-BFGS-B: warm-start + random restarts to escape local minima."""
    best = minimize(lambda p: cost_fn(p, hamiltonian), initial_guess,
                    method='L-BFGS-B', options={'maxiter': 1000, 'ftol': 1e-14})
    for _ in range(n_restarts):
        x0 = best.x + np.random.normal(0, 0.1, len(best.x))
        trial = minimize(lambda p: cost_fn(p, hamiltonian), x0,
                         method='L-BFGS-B', options={'maxiter': 1000, 'ftol': 1e-14})
        if trial.fun < best.fun:
            best = trial
    return best


# Sweep h=2.0 → 0.0: el HVA con |+⟩^N es exacto a h→∞, así que empezamos
# donde θ≈0 ya es óptimo y warm-start propaga hacia h=0.
current_guess = np.random.uniform(-0.05, 0.05, hva_qc.num_parameters)

print(f"Fase 2: VQE multi-start sweep DESCENDENTE (p={p_layers})...")
print("  (|+⟩^N = ground state paramagnético → sweep desde h=2.0 hacia h=0.0)\n")
for idx in reversed(range(len(h_values))):
    h = h_values[idx]
    H = build_tfim_hamiltonian(N, J, h)

    res = multi_start_vqe(H, current_guess, n_restarts=3)

    bound_qc = hva_qc.assign_parameters(res.x)
    sv_ansatz = Statevector(bound_qc)
    fid = state_fidelity(sv_ansatz, Statevector(exact_data[idx]["ground_state"]))
    e_error = abs(res.fun - exact_data[idx]["ground_energy"])

    status = "✅" if fid >= 0.995 else "⚠️"
    print(f"  {status} h={h:.2f}: fid={fid:.6f}, ΔE={e_error:.2e}, nit={res.nit}")

    vqe_results_dict[idx] = {
        "h": h, "theta_opt": res.x.copy(), "energy": res.fun,
        "energy_error": e_error, "fidelity": fid, "n_iters": res.nit,
    }
    current_guess = res.x

# Reordenar por h creciente para persistencia
vqe_results = [vqe_results_dict[i] for i in range(len(h_values))]

# --- Visualización ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(h_values, [r["fidelity"] * 100 for r in vqe_results], 'g^-')
ax1.axhline(y=99.5, color='orange', linestyle=':', label='Umbral 99.5%')
ax1.axvline(x=1.0, color='red', linestyle='--', label='Punto Crítico')
ax1.set_xlabel('h/J'); ax1.set_ylabel('Fidelidad (%)')
ax1.set_title('Calidad del HVA (p=2)'); ax1.legend(); ax1.grid(True)

ax2.semilogy(h_values, [r["energy_error"] for r in vqe_results], 'ro-')
ax2.axvline(x=1.0, color='red', linestyle='--', label='Punto Crítico')
ax2.set_xlabel('h/J'); ax2.set_ylabel('|E_VQE - E_exact|')
ax2.set_title('Error Energético'); ax2.legend(); ax2.grid(True)

plt.tight_layout(); plt.show()

fids = [r["fidelity"] for r in vqe_results]
print("\n--- RESULTADOS FASE 2 ---")
print(f"Fidelidad: promedio={np.mean(fids)*100:.2f}%, mínima={np.min(fids)*100:.2f}%")
print(f"Error energético máximo: {max(r['energy_error'] for r in vqe_results):.2e}")
n_good = sum(1 for f in fids if f >= 0.995)
print(f"Puntos con fid ≥ 99.5%: {n_good}/{len(fids)}")
print("\nNOTA: La fidelidad baja para h < 1.0 es un LÍMITE DE EXPRESIBILIDAD del HVA p=2")
print("con |+⟩^N. El ground state ferromagnético (h→0) requiere más profundidad.")
print("Esto es esperado y consistente con Mele et al. — el PoC es válido para h ≥ 1.0.")

## Celda 6: Persistencia de Datos para Fase 3

## Celda 5b: Diagnóstico de expresibilidad


In [ ]:
# --- Diagnóstico: métricas priorizadas por relevancia física ---
print(f"{'h':>5} | {'ΔE/gap':>8} | {'⟨X⟩':>7} {'⟨ZZ⟩':>7} | {'ΔE':>9} | {'fid':>7} | {'nit':>3}")
print("─" * 72)
h_boundary = None
for idx, r in enumerate(vqe_results):
    h = r['h']
    gap = exact_data[idx]['gap']
    de_gap = (r['energy_error'] / gap * 100) if gap > 1e-10 else float('inf')

    # Observables del ansatz optimizado
    sv = Statevector(hva_qc.assign_parameters(r['theta_opt']))
    mx = np.mean([sv.expectation_value(op).real for op in ops_X])
    zz = np.mean([sv.expectation_value(op).real for op in ops_ZZ])

    ok = "✅" if r['fidelity'] >= 0.995 else "⚠️"
    print(f"{h:5.2f} | {de_gap:7.2f}% | {mx:7.4f} {zz:7.4f} | {r['energy_error']:.2e} | {r['fidelity']:.4f} | {r['n_iters']:3d} {ok}")
    if h_boundary is None and r['fidelity'] >= 0.995:
        h_boundary = h

print(f"\n📊 Régimen válido (fid ≥ 99.5%): h ≥ {h_boundary:.1f}" if h_boundary else "\n❌ Ningún punto alcanza fid ≥ 99.5%")
print(f"   Puntos válidos: {sum(1 for r in vqe_results if r['fidelity'] >= 0.995)}/{len(vqe_results)}")
print("\n   Columnas ordenadas por prioridad física:")
print("   1. ΔE/gap — ¿resolvemos la física? (<5% = aceptable)")
print("   2. ⟨X⟩, ⟨ZZ⟩ — observables locales (lo que mediríamos en hardware)")
print("   3. ΔE — error energético absoluto")
print("   4. fid — fidelidad (solo validación noiseless, prohibido en hardware)")

In [ ]:
np.savez("phase1_phase2_tfim_N6_p2.npz",
    h_values=h_values, J=J, n_qubits=N, p_layers=p_layers,
    ground_energies=np.array([d["ground_energy"] for d in exact_data]),
    gaps=np.array([d["gap"] for d in exact_data]),
    mag_x=np.array([d["mag_x"] for d in exact_data]),
    corr_zz=np.array([d["corr_zz"] for d in exact_data]),
    theta_opt=np.array([r["theta_opt"] for r in vqe_results]),
    vqe_energies=np.array([r["energy"] for r in vqe_results]),
    fidelities=np.array([r["fidelity"] for r in vqe_results]),
)
print("Dataset guardado: phase1_phase2_tfim_N6_p2.npz")